# EA1 — Diseño e implementación de una base de datos analítica

**Big Data (ISD-25)** · Ingeniería de Software y Datos · IU Digital de Antioquia

| | |
|---|---|
| **Grupo** | 23 |
| **Integrantes** | Manuela Tangarife Zapata |
| **Caso de estudio** | Wanderbricks |
| **Fecha de entrega** | domingo 30 de agosto |
| **🎥 Enlace al video** | *(pegar aquí — 6 a 9 minutos, mínimo 3 minutos por integrante)* |

> ⚠️ **Antes de entregar:** verificar que el enlace del video abra desde una cuenta distinta a la propia.
> Un enlace inaccesible se califica como no entregado.

---
## 1. Contexto y problema

Wanderbricks es un marketplace de alquiler vacacional que recopila información sobre usuarios, anfitriones, propiedades, reservas, pagos, reseñas y comportamiento de navegación. La base de datos analítica tiene como objetivo integrar estas fuentes para facilitar el análisis del comportamiento de los usuarios y el desempeño de las propiedades, permitiendo conectar la actividad de navegación con las reservas y los resultados operativos.Con los datos disponibles, la base analítica podrá responder preguntas como: ¿qué destinos y propiedades generan más reservas y mayor valor?, ¿cómo se relacionan las reservas con los pagos?, ¿qué características tienen las propiedades con mayor interración?, y ¿qué vínculo existe entre el comportamiento de navegación de los usuarios y sus reservas?La solución también debe soportar la incorporación de datos variados, incluyendo información estructurada de transacciones, eventos de navegación semiestructurados y texto libre de reseñas y registros de soporte. Por eso, se busca un diseño que conserve los datos originales, los transforme para su análisis y mantenga flexibilidad ante futuros cambios en su estructura.

---
## 2. Descripción de los datos

In [0]:
# Exploración inicial del caso
display(spark.sql("SHOW TABLES IN samples.wanderbricks"))

In [0]:
# Conteo de filas por tabla
for t in [r[1] for r in spark.sql("SHOW TABLES IN samples.wanderbricks").collect()]:
    print(f"{t:30s} {spark.table(f'samples.wanderbricks.{t}').count():>12,}")


## Descripción de las tablas a usar

El caso Wanderbricks está compuesto por 16 tablas que reúnen 1.208.596 registros. Para la implementación de la base analítica se seleccionaron nueve tablas, bookings, payments, users, properties, hosts, destinations, reviews, clickstream y customer_support_logs, que en conjunto reúnen 545.676 registros y concentran las principales entidades, transacciones y eventos necesarios para el análisis planteado.

Los datos incluyen distintos tipos estructurados como long, integer, float, decimal, string, date, timestamp y boolean. También se encontraron estructuras semiestructuradas. En clickstream, el campo metadata es un struct con los campos device y referrer. En customer_support_logs, el campo messages es un array de struct que contiene message, sender, sentiment y timestamp. Además, hay texto libre en campos como reviews.comment, properties.description y los mensajes de soporte.

Las relaciones principales entre tablas se dan por identificadores compartidos. bookings se vincula con users mediante user_id y con properties mediante property_id. payments usa booking_id para relacionarse con las reservas, y properties usa host_id para vincularse con los anfitriones. La validación mostró correspondencia completa entre bookings.user_id y users.user_id, entre bookings.property_id y properties.property_id, y entre properties.host_id y hosts.host_id. Sin embargo, en payments se detectaron 8 valores de booking_id que no tienen correspondencia en bookings, lo que indica una inconsistencia en la integridad referencial.

Sobre la calidad de los datos, la mayoría de las tablas no presentó valores nulos en las columnas analizadas. En users hubo 124 valores nulos en email y 61,967 en company_name. En reviews se identificaron 476 registros con comment y rating nulos. Estos casos deberán tenerse en cuenta durante la transformación de los datos para la capa Silver.

---
## 3. Decisiones de diseño y justificación

| Criterio del caso | Relacional | NoSQL | Lakehouse | Decisión |
|---|---|---|---|---|
| Datos estructurados y relaciones | Muy adecuado para bookings, payments, users y properties, con relaciones claras mediante identificadores. | Puede manejar estos datos, pero las relaciones entre varias entidades pueden requerir mayor complejidad según el modelo elegido. | Permite trabajar con estas relaciones mediante SQL y Spark sobre tablas Delta. | Lakehouse |
| Datos semiestructurados | Las estructuras anidadas de clickstream y customer_support_logs requieren estructuras adicionales o transformaciones. | Es una alternativa natural para documentos y estructuras anidadas. | Permite conservar y transformar struct y array sin perder el acceso a los datos originales. | Lakehouse |
| Texto libre | Puede almacenarse, pero no es una fortaleza específica del modelo relacional. | Puede manejar texto como parte de documentos. | Permite integrar el texto de reviews y soporte con el resto de los datos analíticos. | Lakehouse |
| Integración de múltiples tipos de datos | Requiere diseñar y adaptar estructuras para cada tipo de información. | Flexible, pero puede dificultar una visión analítica integrada de entidades relacionadas. | Integra datos estructurados y semiestructurados en una misma plataforma. | Lakehouse |
| Calidad y transformación de datos | Permite aplicar restricciones y transformaciones. | Flexible, pero la validación depende del modelo y la implementación. | Permite conservar los datos originales en Bronze y aplicar limpieza y reglas en Silver. | Lakehouse |
| Evolución del esquema | Generalmente requiere cambios controlados en las estructuras y tablas. | Presenta alta flexibilidad de esquema. | Delta permite evolucionar el esquema manteniendo el flujo de datos. | Lakehouse |
| Trazabilidad y versiones de datos | Puede implementarse, pero no es una característica central del modelo. | Depende de la tecnología utilizada. | Delta permite evidenciar historial y consultar versiones anteriores mediante Time Travel. | Lakehouse |

# Justificación de la decisión

El modelo relacional es una opción adecuada para gran parte del caso Wanderbricks, especialmente para los datos estructurados como usuarios, reservas, pagos, propiedades y anfitriones. Las relaciones entre estas entidades se definen mediante identificadores compartidos y requieren consultas que unan varias tablas. Sin embargo, el caso también incluye estructuras semiestructuradas y texto libre, lo que hace necesario complementar el modelo con otras estructuras y transformaciones.

El paradigma NoSQL ofrece una ventaja importante para manejar datos semiestructurados. Los modelos documentales permiten almacenar documentos con campos anidados, arreglos y subdocumentos, por lo que estructuras como customer_support_logs.messages, que es un array de struct, podrían representarse de manera natural en este tipo de modelo. Sin embargo, dado que el análisis también depende de las relaciones entre reservas, usuarios, propiedades y pagos, utilizar únicamente un modelo documental no resulta tan conveniente para cubrir todas las necesidades analíticas del caso.

Se eligió el enfoque lakehouse para Wanderbricks porque permite trabajar con diferentes tipos de datos dentro de una misma arquitectura y aplicar transformaciones progresivas. La arquitectura medallion organiza los datos en capas Bronze, Silver y Gold: Bronze conserva los datos en bruto, mientras que Silver se utiliza para limpieza, validación, manejo de valores faltantes, estandarización y combinación de diferentes fuentes. Esto resulta especialmente útil para el caso analizado, donde se encontraron valores nulos y 8 pagos cuyo booking_id no corresponde a ninguna reserva, ya que permite conservar los datos originales y aplicar las reglas de calidad en las capas posteriores.

Además, el uso de tablas Delta proporciona propiedades importantes para la actividad, como transacciones ACID, control y evolución del esquema y versionado de los datos mediante Time Travel. Delta Lake está diseñado para funcionar sobre arquitecturas lakehouse y permite mantener versiones históricas de las tablas, realizar operaciones transaccionales y manejar cambios en el esquema.

Por estas razones, el lakehouse es la opción que mejor se adapta a las características identificadas en Wanderbricks y proporciona una base adecuada para las consultas analíticas y las transformaciones realizadas durante el proyecto.


**Referencias (APA 7):**

Databricks. (2026, 7 de mayo). What is the medallion lakehouse architecture? https://docs.databricks.com/aws/en/lakehouse/medallion

Databricks. (2026, 10 de julio). What is Delta Lake in Databricks? https://docs.databricks.com/aws/en/delta

Delta Lake. (s. f.). Welcome to the Delta Lake documentation. https://docs.delta.io/

MongoDB. (s. f.). Embedded data in your MongoDB schema. https://www.mongodb.com/docs/manual/data-modeling/embedding/

Google. (s. f.). Traductor de google [Servicio de traducción]. https://translate.google.com/

---
## 4. Implementación
### 4.1 Catálogo, esquema y volumen

In [0]:
CATALOGO = "bigdata_grupo23"   
ESQUEMA  = "wanderbricks"
VOLUMEN  = "datos_crudos"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO}")
spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {CATALOGO}.{ESQUEMA}")
spark.sql(f"CREATE VOLUME  IF NOT EXISTS {CATALOGO}.{ESQUEMA}.{VOLUMEN}")

spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql(f"USE SCHEMA {ESQUEMA}")
print(f"Trabajando en {CATALOGO}.{ESQUEMA}")

### 4.2 Capa bronce — ingesta de datos crudos

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    IntegerType,
    FloatType,
    StringType,
    DateType,
    TimestampType,
    BooleanType,
    DoubleType,
    DecimalType,
    ArrayType
)

# Bronze conserva los datos originales por eso todavía no tiene limpieza ni correcciones sobre ellos.

esquemas = {

    "bookings": StructType([
        StructField("booking_id", LongType(), False),
        StructField("user_id", LongType(), True),
        StructField("property_id", LongType(), True),
        StructField("check_in", DateType(), True),
        StructField("check_out", DateType(), True),
        StructField("guests_count", IntegerType(), True),
        StructField("total_amount", FloatType(), True),
        StructField("status", StringType(), True),
        StructField("created_at", TimestampType(), True),
        StructField("updated_at", TimestampType(), True)
    ]),

    "payments": StructType([
        StructField("payment_id", LongType(), False),
        StructField("booking_id", LongType(), True),
        StructField("amount", DecimalType(15, 4), True),
        StructField("payment_method", StringType(), True),
        StructField("status", StringType(), True),
        StructField("payment_date", TimestampType(), True)
    ]),

    "users": StructType([
        StructField("user_id", LongType(), False),
        StructField("email", StringType(), True),
        StructField("name", StringType(), True),
        StructField("country", StringType(), True),
        StructField("user_type", StringType(), True),
        StructField("created_at", TimestampType(), True),
        StructField("is_business", BooleanType(), True),
        StructField("company_name", StringType(), True)
    ]),

    "properties": StructType([
        StructField("property_id", LongType(), False),
        StructField("host_id", LongType(), True),
        StructField("destination_id", LongType(), True),
        StructField("title", StringType(), True),
        StructField("description", StringType(), True),
        StructField("base_price", FloatType(), True),
        StructField("property_type", StringType(), True),
        StructField("max_guests", IntegerType(), True),
        StructField("bedrooms", IntegerType(), True),
        StructField("bathrooms", IntegerType(), True),
        StructField("property_latitude", FloatType(), True),
        StructField("property_longitude", FloatType(), True),
        StructField("created_at", DateType(), True)
    ]),

    "hosts": StructType([
        StructField("host_id", LongType(), False),
        StructField("name", StringType(), True),
        StructField("email", StringType(), True),
        StructField("phone", StringType(), True),
        StructField("is_verified", BooleanType(), True),
        StructField("is_active", BooleanType(), True),
        StructField("rating", FloatType(), True),
        StructField("country", StringType(), True),
        StructField("joined_at", DateType(), True)
    ]),

    "destinations": StructType([
        StructField("destination_id", LongType(), False),
        StructField("destination", StringType(), True),
        StructField("country", StringType(), True),
        StructField("state_or_province", StringType(), True),
        StructField("state_or_province_code", StringType(), True),
        StructField("description", StringType(), True)
    ]),

    "reviews": StructType([
        StructField("booking_id", LongType(), True),
        StructField("comment", StringType(), True),
        StructField("created_at", TimestampType(), True),
        StructField("is_deleted", BooleanType(), True),
        StructField("property_id", LongType(), True),
        StructField("rating", DoubleType(), True),
        StructField("review_id", LongType(), False),
        StructField("updated_at", TimestampType(), True),
        StructField("user_id", LongType(), True)
    ]),

    "clickstream": StructType([
        StructField("event", StringType(), True),

        # metadata es un struct porque contiene dos campos relacionados dentro de una misma columna.
        StructField(
            "metadata",
            StructType([
                StructField("device", StringType(), True),
                StructField("referrer", StringType(), True)
            ]),
            True
        ),

        StructField("property_id", LongType(), True),
        StructField("timestamp", TimestampType(), True),
        StructField("user_id", LongType(), True)
    ]),

    "customer_support_logs": StructType([
        StructField("created_at", StringType(), True),

        # Messages es un array porque una fila puede contener varios mensajes de soporte.
        StructField(
            "messages",
            ArrayType(
                StructType([
                    StructField("message", StringType(), True),
                    StructField("sender", StringType(), True),
                    StructField("sentiment", StringType(), True),
                    StructField("timestamp", StringType(), True)
                ]),
                True
            ),
            True
        ),

        StructField("support_agent_id", StringType(), True),
        StructField("ticket_id", StringType(), False),
        StructField("user_id", LongType(), True)
    ])
}

# Tablas que llevo a bronze
# Las mantengo iguales a las seleccionadas en la sección 2
tablas = [
    "bookings",
    "payments",
    "users",
    "properties",
    "hosts",
    "destinations",
    "reviews",
    "clickstream",
    "customer_support_logs"
]

for tabla in tablas:
    # Aquí leeo la tabla original de wanderbricks.
    df = spark.table(f"samples.wanderbricks.{tabla}")

    # Aplico el esquema que definí arriba.
    columnas = [
        F.col(campo.name).cast(campo.dataType).alias(campo.name)
        for campo in esquemas[tabla].fields
    ]

    df = df.select(columnas)

    # Guardo una copia en formato delta dentro del catálogo.
    tabla_bronze = f"{tabla}_bronze"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(tabla_bronze)
    )

    print(f"Creada: {tabla_bronze}")

In [0]:
# Verifico que las tablas Bronze quedaron creadas correctamente.
display(spark.sql("SHOW TABLES IN bigdata_grupo23.wanderbricks"))

### 4.3 Capa plata — datos limpios y tipados

In [0]:
from pyspark.sql import functions as F

# Con esto tomo la tabla bronze de reservas para empezar a preparar la capa silver
bookings = spark.table("bookings_bronze")

# Limpio los textos y calculo la duración de cada reserva en días.
bookings_silver = (
    bookings
    .withColumn("status", F.trim(F.lower(F.col("status"))))
    .withColumn(
        "reservation_days",
        F.datediff(F.col("check_out"), F.col("check_in"))
    )
)

# Guardo la tabla limpia en formato delta.
(
    bookings_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("bookings_silver")
)

print("Creada: bookings_silver")

In [0]:
# Tomo la tabla bronze de pagos
payments = spark.table("payments_bronze")

# Tomo las reservas silver para comprobar que cada pago tenga una reserva asociada.
bookings_validas = (
    spark.table("bookings_silver")
    .select("booking_id")
    .distinct()
)

# Conservo únicamente los pagos que tienen una reserva válida y normalizo el método y estado del pago.
payments_silver = (
    payments
    .join(
        bookings_validas,
        on="booking_id",
        how="inner"
    )
    .withColumn("payment_method", F.trim(F.lower(F.col("payment_method"))))
    .withColumn("status", F.trim(F.lower(F.col("status"))))
)

# Guardo la tabla limpia en formato delta.
(
    payments_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("payments_silver")
)

print("Creada: payments_silver")

In [0]:
# Tomo la tabla bronze de usuarios
users = spark.table("users_bronze")

# Limpio los espacios de los textos y normalizo el correo y Los valores nulos se conservan porque no necesariamente representan un error.
users_silver = (
    users
    .withColumn("name", F.trim(F.col("name")))
    .withColumn("country", F.trim(F.col("country")))
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn("company_name", F.trim(F.col("company_name")))
)

# Guardo la tabla limpia en formato delta.
(
    users_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("users_silver")
)

print("Creada: users_silver")

In [0]:
# Tomo la tabla bronze de propiedades
properties = spark.table("properties_bronze")

# Limpio los textos y conservo los valores numéricos para los análisis y también elimino espacios innecesarios en los nombres y descripciones.
properties_silver = (
    properties
    .withColumn("title", F.trim(F.col("title")))
    .withColumn("description", F.trim(F.col("description")))
    .withColumn("property_type", F.trim(F.lower(F.col("property_type"))))
)

# Guardo la tabla limpia en formato delta.
(
    properties_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("properties_silver")
)

print("Creada: properties_silver")

In [0]:
# Tomo la tabla bronze de anfitriones.
hosts = spark.table("hosts_bronze")

# Limpio los campos de texto y normalizo el país.
# No elimino registros porque no hay problemas de completitud en esta tabla.
hosts_silver = (
    hosts
    .withColumn("name", F.trim(F.col("name")))
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn("phone", F.trim(F.col("phone")))
    .withColumn("country", F.trim(F.col("country")))
)

# Guardo la tabla limpia en formato delta.
(
    hosts_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("hosts_silver")
)

print("Creada: hosts_silver")

In [0]:
# Tomo la tabla bronze de destinos
destinations = spark.table("destinations_bronze")

# Limpio los nombres geográficos para mantener una escritura uniforme
destinations_silver = (
    destinations
    .withColumn("destination", F.trim(F.col("destination")))
    .withColumn("country", F.trim(F.col("country")))
    .withColumn("state_or_province", F.trim(F.col("state_or_province")))
    .withColumn("state_or_province_code", F.upper(F.trim(F.col("state_or_province_code"))))
    .withColumn("description", F.trim(F.col("description")))
)

# Guardo la tabla limpia en formato delta.
(
    destinations_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("destinations_silver")
)

print("Creada: destinations_silver")

In [0]:
# Tomo la tabla bronze de reseñas.
reviews = spark.table("reviews_bronze")

# Mantengo las reseñas activas y con una calificación disponible y los registros originales permanecen en bronze para conservar la tranzabilidad.
reviews_silver = (
    reviews
    .filter(F.col("is_deleted") == False)
    .filter(F.col("rating").isNotNull())
    .withColumn("comment", F.trim(F.col("comment")))
)

# Guardo la tabla limpia en formato delta.
(
    reviews_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("reviews_silver")
)

print("Creada: reviews_silver")

In [0]:
from pyspark.sql import functions as F

# Tomo la tabla bronze de eventos de navegación.
clickstream = spark.table("bigdata_grupo23.wanderbricks.clickstream_bronze")

# Extraigo los campos que estaban dentro del struct metadata para que puedan utilizarse más fácilmente en consultas analíticas.
clickstream_silver = (
    clickstream
    .withColumn("device", F.col("metadata.device"))
    .withColumn("referrer", F.col("metadata.referrer"))
    .withColumn("event", F.trim(F.col("event")))
    .drop("metadata")
)

# Guardo la tabla transformada en formato delta
(
    clickstream_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("bigdata_grupo23.wanderbricks.clickstream_silver")
)

print("Creada: clickstream_silver")

In [0]:
# Tomo la tabla bronze de los registros de soporte
support = spark.table(
    "bigdata_grupo23.wanderbricks.customer_support_logs_bronze"
)

# Cada ticket puede contener varios mensajes dentro del array "messages".
# Con explode se convierte cada mensaje en una fila independiente
support_silver = (
    support
    .withColumn("message_data", F.explode("messages"))
    .select(
        "ticket_id",
        "user_id",
        "support_agent_id",
        "created_at",
        F.col("message_data.message").alias("message"),
        F.col("message_data.sender").alias("sender"),
        F.col("message_data.sentiment").alias("sentiment"),
        F.col("message_data.timestamp").alias("message_timestamp")
    )
)

# Guardo los mensajes individuales en formato delta.
(
    support_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "bigdata_grupo23.wanderbricks.customer_support_logs_silver"
    )
)

print("Creada: customer_support_logs_silver")

### 4.4 Datos semiestructurados


La información semiestructurada del caso se concentra principalmente en las tablas clickstream y customer_support_logs. En clickstream, la columna metadata es un struct que incluye los campos device y referrer. En customer_support_logs, la columna messages es un array de struct, donde cada elemento contiene información del mensaje, remitente, sentimiento y fecha.En la capa Bronze, estas estructuras se conservaron sin cambios para mantener la representación original de los datos. Durante la construcción de la capa Silver, se transformaron para facilitar su análisis. En clickstream, los campos device y referrer se extrajeron de metadata como columnas independientes. En customer_support_logs, se aplicó explode() para convertir los elementos del array messages en filas separadas.

In [0]:
# pequeña muestra de la estructura original de clickstream.
# En bronze se conserva metadata como un struct.
display(
    spark.table(
        "bigdata_grupo23.wanderbricks.clickstream_bronze"
    ).select(
        "event",
        "metadata",
        "property_id",
        "timestamp",
        "user_id"
    ).limit(5)
)

In [0]:
# Muestro como queda la misma información después de la transformación.
# Los campos del struct metadata ahora son columnas independientes.
display(
    spark.table(
        "bigdata_grupo23.wanderbricks.clickstream_silver"
    ).select(
        "event",
        "device",
        "referrer",
        "property_id",
        "timestamp",
        "user_id"
    ).limit(5)
)

### 4.5 Propiedades del lakehouse

### ACID

In [0]:
# Modifico de forma controlada una reserva para generar una nueva versión delta.
# La operación queda registrada en el historial de la tabla.
spark.sql("""
    UPDATE bigdata_grupo23.wanderbricks.bookings_silver
    SET reservation_days = reservation_days
    WHERE booking_id = 1
""")

print("Operación transaccional ejecutada sobre bookings_silver.")

In [0]:
# Hago una modificación controlada sobre la tabla silver.
# Cambio el estado de una reserva existente para generar una nueva versión delta.
spark.sql("""
    UPDATE bigdata_grupo23.wanderbricks.bookings_silver
    SET status = 'revision'
    WHERE booking_id = (
        SELECT MIN(booking_id)
        FROM bigdata_grupo23.wanderbricks.bookings_silver
    )
""")

print("Operación transaccional ejecutada sobre bookings_silver.")

### Time Travel

In [0]:
# Muestro el historial de cambios de bookings_silver.
# Aquí se pueden ver las versiones que delta ha registrado.
display(
    spark.sql("""
        DESCRIBE HISTORY bigdata_grupo23.wanderbricks.bookings_silver
    """)
)

In [0]:
# Consulto la versión anterior de bookings_silver mediante time travel.
# La versión 0 representa la tabla antes de la modificación realizada
display(
    spark.sql("""
        SELECT *
        FROM bigdata_grupo23.wanderbricks.bookings_silver VERSION AS OF 0
        WHERE booking_id = (
            SELECT MIN(booking_id)
            FROM bigdata_grupo23.wanderbricks.bookings_silver
        )
    """)
)

### Evolución de esquema con mergeSchema

In [0]:
from pyspark.sql import functions as F

# Agrego una nueva columna para demostrar que el esquema puede evolucionar.
# La información original de la tabla no se elimina.
nueva_columna = (
    spark.table("bigdata_grupo23.wanderbricks.bookings_silver")
    .withColumn("data_source", F.lit("Wanderbricks"))
)

# Escribo la tabla nuevamente permitiendo la evolución del esquema.
(
    nueva_columna.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .saveAsTable("bigdata_grupo23.wanderbricks.bookings_silver")
)

print("Esquema actualizado con la nueva columna: data_source")

In [0]:
# Verifico que la nueva columna haya quedado incorporada al esquema.
# Ahora bookings_silver tiene la columna data_source.
display(
    spark.table(
        "bigdata_grupo23.wanderbricks.bookings_silver"
    ).select(
        "booking_id",
        "status",
        "reservation_days",
        "data_source"
    ).limit(5)
)

### 4.6 Consultas analíticas

In [0]:
# Consulta 1 — pregunta que responde: ¿Qué destinos generan mayor valor de reservas?

In [0]:
%sql

SELECT
    d.destination,
    d.country,
    COUNT(b.booking_id) AS total_reservas,
    ROUND(SUM(b.total_amount), 2) AS valor_total_reservas,
    ROUND(AVG(b.total_amount), 2) AS valor_promedio_reserva
FROM bigdata_grupo23.wanderbricks.bookings_silver AS b
INNER JOIN bigdata_grupo23.wanderbricks.properties_silver AS p
    ON b.property_id = p.property_id
INNER JOIN bigdata_grupo23.wanderbricks.destinations_silver AS d
    ON p.destination_id = d.destination_id
GROUP BY
    d.destination,
    d.country
ORDER BY
    valor_total_reservas DESC;

In [0]:
from pyspark.sql import functions as F

# Repito la consulta anterior utilizando PySpark.
# Primero resumo las interacciones y las reservas por propiedad para evitar duplicar registros al momento de hacer el JOIN.

clickstream = spark.table(
    "bigdata_grupo23.wanderbricks.clickstream_silver"
)

bookings = spark.table(
    "bigdata_grupo23.wanderbricks.bookings_silver"
)

properties = spark.table(
    "bigdata_grupo23.wanderbricks.properties_silver"
)

interacciones = (
    clickstream
    .filter(F.col("property_id").isNotNull())
    .groupBy("property_id")
    .agg(
        F.count("*").alias("total_interacciones")
    )
)

reservas = (
    bookings
    .filter(F.col("property_id").isNotNull())
    .groupBy("property_id")
    .agg(
        F.count("booking_id").alias("total_reservas"),
        F.round(F.sum("total_amount"), 2).alias("valor_total_reservas")
    )
)

resultado = (
    interacciones
    .join(
        properties,
        interacciones.property_id == properties.property_id,
        "inner"
    )
    .join(
        reservas,
        interacciones.property_id == reservas.property_id,
        "left"
    )
    .select(
        properties.property_id,
        properties.title,
        "total_interacciones",
        F.coalesce("total_reservas", F.lit(0)).alias("total_reservas"),
        F.coalesce(
            "valor_total_reservas",
            F.lit(0)
        ).alias("valor_total_reservas")
    )
    .orderBy(F.col("total_interacciones").desc())
    .limit(10)
)

display(resultado)

In [0]:
# Consulta 2 — pregunta que responde: ¿Qué propiedades reciben mayor interacción y cuántas reservas generan?

In [0]:
from pyspark.sql import functions as F

# Repito la consulta anterior utilizando PySpark.
# La idea es obtener el mismo resultado, pero construyendo la consulta con DataFrames.

bookings = spark.table("bigdata_grupo23.wanderbricks.bookings_silver")
properties = spark.table("bigdata_grupo23.wanderbricks.properties_silver")
destinations = spark.table("bigdata_grupo23.wanderbricks.destinations_silver")

resultado = (
    bookings
    .join(properties, bookings.property_id == properties.property_id, "inner")
    .join(
        destinations,
        properties.destination_id == destinations.destination_id,
        "inner"
    )
    .groupBy(
        destinations.destination,
        destinations.country
    )
    .agg(
        F.count(bookings.booking_id).alias("total_reservas"),
        F.round(F.sum(bookings.total_amount), 2).alias("valor_total_reservas"),
        F.round(F.avg(bookings.total_amount), 2).alias("valor_promedio_reserva")
    )
    .orderBy(F.col("valor_total_reservas").desc())
    .limit(10)
)

display(resultado)

In [0]:
%sql

WITH interacciones AS (
    SELECT
        property_id,
        COUNT(*) AS total_interacciones
    FROM bigdata_grupo23.wanderbricks.clickstream_silver
    WHERE property_id IS NOT NULL
    GROUP BY property_id
),
reservas AS (
    SELECT
        property_id,
        COUNT(booking_id) AS total_reservas,
        ROUND(SUM(total_amount), 2) AS valor_total_reservas
    FROM bigdata_grupo23.wanderbricks.bookings_silver
    WHERE property_id IS NOT NULL
    GROUP BY property_id
)
SELECT
    p.property_id,
    p.title,
    i.total_interacciones,
    COALESCE(r.total_reservas, 0) AS total_reservas,
    COALESCE(r.valor_total_reservas, 0) AS valor_total_reservas
FROM interacciones AS i
INNER JOIN bigdata_grupo23.wanderbricks.properties_silver AS p
    ON i.property_id = p.property_id
LEFT JOIN reservas AS r
    ON i.property_id = r.property_id
ORDER BY total_interacciones DESC
LIMIT 10;

In [0]:
# Consulta 3 — pregunta que responde: ¿Qué métodos de pago son más utilizados y qué valor representan?

In [0]:
%sql

SELECT
    payment_method,
    COUNT(payment_id) AS total_pagos,
    ROUND(SUM(amount), 2) AS valor_total_pagado,
    ROUND(AVG(amount), 2) AS valor_promedio_pago
FROM bigdata_grupo23.wanderbricks.payments_silver
WHERE status = 'completed'
GROUP BY payment_method
ORDER BY valor_total_pagado DESC;

In [0]:
from pyspark.sql import functions as F

# Repito la consulta anterior utilizando PySpark.
# Solo considero pagos completados para medir el valor realmente procesado.

payments = spark.table(
    "bigdata_grupo23.wanderbricks.payments_silver"
)

resultado = (
    payments
    .filter(F.col("status") == "completed")
    .groupBy("payment_method")
    .agg(
        F.count("payment_id").alias("total_pagos"),
        F.round(F.sum("amount"), 2).alias("valor_total_pagado"),
        F.round(F.avg("amount"), 2).alias("valor_promedio_pago")
    )
    .orderBy(F.col("valor_total_pagado").desc())
)

display(resultado)

In [0]:
# Consulta 4 — pregunta que responde: ¿Qué relación existe entre la calificación promedio de una propiedad y la cantidad de reservas que genera?

In [0]:
%sql

WITH valoraciones AS (
    SELECT
        property_id,
        ROUND(AVG(rating), 2) AS calificacion_promedio,
        COUNT(review_id) AS total_resenas
    FROM bigdata_grupo23.wanderbricks.reviews_silver
    WHERE property_id IS NOT NULL
    GROUP BY property_id
),
reservas AS (
    SELECT
        property_id,
        COUNT(booking_id) AS total_reservas,
        ROUND(SUM(total_amount), 2) AS valor_total_reservas
    FROM bigdata_grupo23.wanderbricks.bookings_silver
    WHERE property_id IS NOT NULL
    GROUP BY property_id
)
SELECT
    p.property_id,
    p.title,
    v.calificacion_promedio,
    v.total_resenas,
    COALESCE(r.total_reservas, 0) AS total_reservas,
    COALESCE(r.valor_total_reservas, 0) AS valor_total_reservas
FROM valoraciones AS v
INNER JOIN bigdata_grupo23.wanderbricks.properties_silver AS p
    ON v.property_id = p.property_id
LEFT JOIN reservas AS r
    ON v.property_id = r.property_id
ORDER BY
    calificacion_promedio DESC,
    total_reservas DESC
LIMIT 10;

In [0]:
from pyspark.sql import functions as F

# Repito la consulta anterior utilizando PySpark.
# Primero calculo la calificación promedio y la cantidad de reseñas por propiedad y luego relaciono esos datos con las reservas.

reviews = spark.table(
    "bigdata_grupo23.wanderbricks.reviews_silver"
)

bookings = spark.table(
    "bigdata_grupo23.wanderbricks.bookings_silver"
)

properties = spark.table(
    "bigdata_grupo23.wanderbricks.properties_silver"
)

valoraciones = (
    reviews
    .filter(F.col("property_id").isNotNull())
    .groupBy("property_id")
    .agg(
        F.round(F.avg("rating"), 2).alias("calificacion_promedio"),
        F.count("review_id").alias("total_resenas")
    )
)

reservas = (
    bookings
    .filter(F.col("property_id").isNotNull())
    .groupBy("property_id")
    .agg(
        F.count("booking_id").alias("total_reservas"),
        F.round(F.sum("total_amount"), 2).alias("valor_total_reservas")
    )
)

resultado = (
    valoraciones
    .join(
        properties,
        valoraciones.property_id == properties.property_id,
        "inner"
    )
    .join(
        reservas,
        valoraciones.property_id == reservas.property_id,
        "left"
    )
    .select(
        properties.property_id,
        properties.title,
        "calificacion_promedio",
        "total_resenas",
        F.coalesce("total_reservas", F.lit(0)).alias("total_reservas"),
        F.coalesce(
            "valor_total_reservas",
            F.lit(0)
        ).alias("valor_total_reservas")
    )
    .orderBy(
        F.col("calificacion_promedio").desc(),
        F.col("total_reservas").desc()
    )
    .limit(10)
)

display(resultado)

In [0]:
# Consulta 5 — pregunta que responde: ¿Desde qué dispositivos se concentra la actividad de los usuarios?

In [0]:
%sql

SELECT
    device,
    COUNT(*) AS total_eventos,
    COUNT(DISTINCT user_id) AS usuarios_unicos,
    ROUND(
        COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (),
        2
    ) AS porcentaje_eventos
FROM bigdata_grupo23.wanderbricks.clickstream_silver
WHERE device IS NOT NULL
GROUP BY device
ORDER BY total_eventos DESC;

In [0]:
from pyspark.sql import functions as F

# Repito la consulta anterior utilizando PySpark.
# Agrupo los eventos por dispositivo y calculo usuarios únicos
# y la participación de cada dispositivo sobre el total.

clickstream = spark.table(
    "bigdata_grupo23.wanderbricks.clickstream_silver"
)

resultado = (
    clickstream
    .filter(F.col("device").isNotNull())
    .groupBy("device")
    .agg(
        F.count("*").alias("total_eventos"),
        F.countDistinct("user_id").alias("usuarios_unicos")
    )
)

total_eventos = clickstream.filter(
    F.col("device").isNotNull()
).count()

resultado = (
    resultado
    .withColumn(
        "porcentaje_eventos",
        F.round(
            F.col("total_eventos") * 100 / total_eventos,
            2
        )
    )
    .orderBy(F.col("total_eventos").desc())
)

display(resultado)

---
## 5. Resultados

Consulta 1: Destinos con mayor valor de reservas

Phuket es el destino que genera el mayor valor de reservas, con 7.292 reservas y aproximadamente 4,06 millones en valor agregado. Gold Coast y Mallorca ocupan el segundo y tercer lugar. Los valores promedio por reserva son bastante similares, por lo que la diferencia en el valor total se debe principalmente a la cantidad de reservas.

Consulta 2: Interacción y reservas por propiedad

La propiedad 4636, ubicada en Gold Coast, presenta el mayor número de interacciones, con 21 eventos, pero no registra reservas. Por otro lado, la propiedad 16326, ubicada en Sardinia, tiene 15 interacciones y 6 reservas por un valor total de 3.642,66. Esto muestra que tener más interacciones no significa necesariamente tener más reservas.

Consulta 3: Métodos de pago más utilizados

Los cinco métodos de pago tienen cantidades de transacciones bastante similares. credit_card es el más utilizado, con 9.168 pagos, y también representa el mayor valor procesado, con aproximadamente 5,08 millones. bank_transfer tiene el mayor valor promedio por pago, con 560,57. En general, los pagos están distribuidos de manera equilibrada entre los diferentes métodos.

Consulta 4: Calificación y cantidad de reservas

Las propiedades con mayor calificación mostradas tienen un promedio de 5,0, pero presentan diferentes cantidades de reservas. Además, cada una tiene solamente una reseña. Por esto, con estos datos no se puede establecer una relación clara entre una calificación alta y un mayor número de reservas. También es importante tener en cuenta la cantidad de reseñas antes de interpretar la calificación.

Consulta 5: Dispositivos utilizados para navegar

El uso de los tres dispositivos está bastante equilibrado. Las tablets registran el mayor número de eventos, con 33.753 y un 33,75% del total. Los computadores de escritorio representan el 33,30% y los dispositivos móviles el 32,94%. Esto muestra que no existe una diferencia significativa en la navegación según el tipo de dispositivo.

---
## 6. Conclusiones

El desarrollo de las consultas permitió confirmar que la información del dataset puede usarse para responder preguntas reales del negocio y conectar diferentes aspectos de la operación, como reservas, interacciones, pagos, calificaciones y dispositivos.

Lo que funcionó: Las consultas revelaron diferencias claras entre destinos, mostraron que no todas las interacciones terminan en reservas y evidenciaron que los métodos de pago tienen un comportamiento bastante equilibrado. También fue posible comparar las calificaciones de las propiedades con sus reservas.

Lo que no funcionó o tuvo limitaciones: Durante el desarrollo, fue necesario revisar y ajustar algunas consultas para evitar resultados engañosos al relacionar tablas con diferentes niveles de detalle. Por ejemplo, en el análisis de calificaciones y reservas, fue importante calcular primero los indicadores por propiedad y luego hacer la unión, para evitar duplicar registros y distorsionar los promedios. Además, algunos resultados no fueron suficientes para establecer relaciones claras, como en el caso de calificaciones de 5.0 basadas solo en una reseña. Esto muestra que no todos los indicadores pueden interpretarse de forma aislada.

Qué haríamos distinto: Si empezáramos de nuevo, consideraría desde el principio variables adicionales para complementar los análisis, especialmente la cantidad de reseñas al estudiar las calificaciones. Además, buscaría incluir más información sobre el comportamiento de los usuarios para entender mejor por qué algunas propiedades reciben muchas interacciones pero pocas o ninguna reserva.

---
## 7. Reparto del trabajo y uso de IA

| Integrante | De qué se encargó | Qué sustenta en el video |
|---|---|---|
| Manuela Tangarife Zapata | Toda la evidencia | |

**Uso de asistentes de IA:** Durante el desarrollo del proyecto utilice ChatGPT como herramienta de apoyo en distintas etapas. Su uso se centró principalmente en la construcción y comprensión de las consultas SQL y PySpark, así como en la revisión de la lógica aplicada.

En las consultas de negocio, la IA ayudó a definir la estructura de las consultas, identificar las operaciones necesarias y proponer formas de hacer las mismas consultas en SQL y PySpark. También fue útil para entender funciones de PySpark como filter, groupBy, agg, join, avg, sum, count y orderBy.

Un caso relevante fue la consulta 4, donde se analizó la relación entre la calificación promedio de las propiedades y su cantidad de reservas. Durante su desarrollo, se revisó cómo realizar el JOIN entre reseñas y reservas para evitar que la combinación de registros afectara el cálculo del promedio. Por eso, se decidió calcular primero las métricas por tabla y luego unir los resultados.

> Este reparto debe coincidir con lo que cada persona demuestra en el video y con el
> historial de commits del repositorio. Las tres fuentes se contrastan al calificar.

---
## 📹 Preguntas obligatorias de sustentación

Cada integrante responde estas tres preguntas en su intervención del video:

1. ¿Por qué eligieron este modelo de datos y qué alternativa descartaron?
2. Muestre una consulta que usted escribió y explique qué hace Spark al ejecutarla.
3. ¿Qué tendrían que cambiar en su diseño si el volumen se multiplicara por cien?

---
## ✅ Antes de entregar

- [ ] El notebook corre completo de arriba abajo sin errores
- [ ] Las siete secciones están diligenciadas, no quedaron textos de plantilla
- [ ] El enlace del video está en la portada y abre desde otra cuenta
- [ ] Todos los integrantes aparecen en el video con cámara al presentarse
- [ ] El notebook está confirmado en el repositorio, en la carpeta /ea1
- [ ] El HTML con salidas visibles está subido a Canvas